# Rule-based NER

### 1.Download Libraries

In [ ]:
!pip install python-docx -q
!pip install levenshtein -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.7 MB/s eta 0:00:00


### 2.Load the Document

In [ ]:
from docx import Document

# --- Load the document ---
DOC_PATH = r"/content/ZF4894_ALV_07Aug2026_physical.docx"
doc = Document(DOC_PATH)

# --- Print paragraphs (plain text blocks) ---
print("=== PARAGRAPHS ===")
for i, para in enumerate(doc.paragraphs):
    if para.text.strip():
        print(f"  [{i}] {para.text}")

# --- Print tables ---
print(f"\n=== TABLES ({len(doc.tables)} found) ===")
for t_idx, table in enumerate(doc.tables):
    print(f"\nTable {t_idx}:")
    for row in table.rows:
        cells = [c.text.strip() for c in row.cells]
        if any(cells):  # skip fully empty rows
            print(f"  {cells}")

=== PARAGRAPHS ===
  [0] ZF4894, ALV, physical
  [1] Barrier 75%, 07 August 2026

=== TABLES (3 found) ===

Table 0:
  ['Party A', 'BANK ABC']
  ['Party B', 'CACIB']
  ['Trade Date', '31 January 2025']
  ['Trade Time', '09:12:15']
  ['Initial Valuation Date', '31 January 2025']
  ['Effective Date', '07 February 2025']
  ['Notional Amount (N)', 'EUR 1 million']
  ['Upfront Payment', '***TBD***% to be paid by Party B to BANK ABC on the Effective Date']
  ['Valuation Date', '31 July 2026']
  ['Termination Date', '07 August 2026']
  ['Underlying', 'Allianz SE (ISIN DE0008404005, Reuters: ALVG.DE)']
  ['Exchange', 'XETRA']
  ['Coupon (C)', '0%']
  ['Barrier (B)', '75.00% of Shareini']

Table 1:
  ['Interest Payments', 'Interest Payments']
  ['Interest Payments', 'None']

Table 2:
  ['Equity Payments', 'Equity Payments']
  ['Initial Price (Shareini)', 'Official closing price of the Underlying on the Initial Valuation Date on the Exchange']
  ['Sharefinal', 'Official closing price of the Unde

### 3. Define Lookup Key Value Mappings

In [ ]:
import re
import json

# Map the label text found in the DOCX
FIELD_MAP = {
    "Counterparty":           ["party a"],
    "Party B":                ["party b"],
    "Trade Date":             ["trade date"],
    "Trade Time":             ["trade time"],
    "Initial Valuation Date": ["initial valuation date"],
    "Effective Date":         ["effective date"],
    "Valuation Date":         ["valuation date"],
    "Maturity":               ["termination date"],
    "Termination Time":       ["termination time"],
    "Notional":               ["notional amount (n)", "notional amount"],
    "Upfront Payment":        ["upfront payment"],
    "Underlying":             ["underlying"],
    "Exchange":               ["exchange"],
    "Coupon":                 ["coupon (c)", "coupon"],
    "Barrier":                ["barrier (b)", "barrier"],
    "Interest Payments":      ["interest payments"],
    "Initial Price":          ["initial price (shareini)"],
    "Final Price":            ["sharefinal"],
    "Future Price Valuation": ["future price valuation"],
    "Calendar":               ["business day"],
    "Calculation Agent":      ["calculation agent"],
    "ISDA Documentation":     ["isda documentation"],
}


# These are the 9 target entities requested
TARGET_ENTITIES = [
    "Counterparty",
    "Initial Valuation Date",
    "Notional",
    "Valuation Date",
    "Maturity",
    "Underlying",
    "Coupon",
    "Barrier",
    "Calendar",
]

print("Field map loaded:", len(FIELD_MAP), "rules defined")
print("Target entities:", TARGET_ENTITIES)

Field map loaded: 22 rules defined
Target entities: ['Counterparty', 'Initial Valuation Date', 'Notional', 'Valuation Date', 'Maturity', 'Underlying', 'Coupon', 'Barrier', 'Calendar']


### 4. Extract Enitites from table [Fallback to Levenshtein Matching]

In [ ]:

from Levenshtein import distance as levenshtein_distance

LABEL_LOOKUP = {
    variant: entity
    for entity, variants in FIELD_MAP.items()
    for variant in variants
}

ALL_VARIANTS = list(LABEL_LOOKUP.keys())


def fuzzy_match(key, max_distance=2):
    # Exact match first
    if key in LABEL_LOOKUP:
        return LABEL_LOOKUP[key], key, False, 0

    # Levenshtein fallback — find closest variant within max_distance
    best_variant  = None
    best_distance = float("inf")

    for variant in ALL_VARIANTS:
        dist = levenshtein_distance(key, variant)
        if dist < best_distance:
            best_distance = dist
            best_variant  = variant

    if best_distance <= max_distance:
        entity_name = LABEL_LOOKUP[best_variant]
        return entity_name, best_variant, True, best_distance

    return None, None, False, None


def extract_from_tables(doc):
    entities = {}

    for table in doc.tables:
        for row in table.rows:
            cells = [cell.text.strip() for cell in row.cells]

            if len(cells) < 2 or not cells[0] or not cells[1]:
                continue
            if cells[0] == cells[1]:
                continue

            key   = cells[0].strip().lower()
            value = cells[1].strip()

            entity_name, matched_variant, is_fuzzy, dist = fuzzy_match(key)

            if entity_name and entity_name in TARGET_ENTITIES:
                if entity_name not in entities:
                    entities[entity_name] = value
                    if is_fuzzy:
                        print(f"  FUZZY  : '{cells[0]}' ≈ '{matched_variant}' (distance={dist}) → {entity_name} = '{value}'")
                    else:
                        print(f"  EXACT  : '{cells[0]}' → {entity_name} = '{value}'")

    return entities


In [ ]:
entities = extract_from_tables(doc)
print(entities)

  EXACT  : 'Party A' → Counterparty = 'BANK ABC'
  EXACT  : 'Initial Valuation Date' → Initial Valuation Date = '31 January 2025'
  EXACT  : 'Notional Amount (N)' → Notional = 'EUR 1 million'
  EXACT  : 'Valuation Date' → Valuation Date = '31 July 2026'
  EXACT  : 'Termination Date' → Maturity = '07 August 2026'
  EXACT  : 'Underlying' → Underlying = 'Allianz SE (ISIN DE0008404005, Reuters: ALVG.DE)'
  EXACT  : 'Coupon (C)' → Coupon = '0%'
  EXACT  : 'Barrier (B)' → Barrier = '75.00% of Shareini'
  EXACT  : 'Business Day' → Calendar = 'TARGET'
{'Counterparty': 'BANK ABC', 'Initial Valuation Date': '31 January 2025', 'Notional': 'EUR 1 million', 'Valuation Date': '31 July 2026', 'Maturity': '07 August 2026', 'Underlying': 'Allianz SE (ISIN DE0008404005, Reuters: ALVG.DE)', 'Coupon': '0%', 'Barrier': '75.00% of Shareini', 'Calendar': 'TARGET'}


### 5. Output Validation and Generation

In [ ]:

SCHEMA = {
    "Counterparty": {
        "type":     "string",
        "format":   "Bank or entity name",
        "example":  "BANK ABC",
        "required": True,
        "nullable": False,
        "pattern":  None,
    },
    "Initial Valuation Date": {
        "type":     "date",
        "format":   "DD Month YYYY",
        "example":  "31 January 2025",
        "required": True,
        "nullable": False,
        "pattern":  r"^\d{1,2}\s+(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4}$",
    },
    "Notional": {
        "type":     "string",
        "format":   "CCY Amount",
        "example":  "EUR 1 million",
        "required": True,
        "nullable": False,
        "pattern":  None,
    },
    "Valuation Date": {
        "type":     "date",
        "format":   "DD Month YYYY",
        "example":  "31 July 2026",
        "required": True,
        "nullable": False,
        "pattern":  r"^\d{1,2}\s+(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4}$",
    },
    "Maturity": {
        "type":     "date",
        "format":   "DD Month YYYY",
        "example":  "07 August 2026",
        "required": True,
        "nullable": False,
        "pattern":  r"^\d{1,2}\s+(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4}$",
    },
    "Underlying": {
        "type":     "string",
        "format":   "Name (ISIN <code>, Reuters: <ticker>)",
        "example":  "Allianz SE (ISIN DE0008404005, Reuters: ALVG.DE)",
        "required": True,
        "nullable": False,
        "pattern":  None,
    },
    "Coupon": {
        "type":     "string",
        "format":   "Percentage or expression",
        "example":  "0%",
        "required": False,
        "nullable": True,
        "pattern":  r"^\d+(\.\d+)?%$",
    },
    "Barrier": {
        "type":     "string",
        "format":   "Percentage of initial price",
        "example":  "75.00% of Shareini",
        "required": False,
        "nullable": True,
        "pattern":  r"^\d+(\.\d+)?%",
    },
    "Calendar": {
        "type":     "string",
        "format":   "Calendar code",
        "example":  "TARGET",
        "required": False,
        "nullable": True,
        "pattern":  None,
    },
}



In [ ]:

from pathlib import Path

DOC_PATH = '/content/ZF4894_ALV_07Aug2026_physical.docx'

def build_output(entities, doc_path):
    result = {}
    errors = []

    for field, rules in SCHEMA.items():
        value = entities.get(field, None)

        if value is None and rules["required"]:
            errors.append(f"MISSING required field: '{field}'")

        if value is not None and rules["pattern"]:
            if not re.match(rules["pattern"], value):
                errors.append(
                    f"FORMAT ERROR: '{field}' = '{value}' "
                    f"does not match expected format '{rules['format']}'"
                )

        result[field] = value

    output = {
        "source":        Path(doc_path).name,
        "engine":        "rule_based_parser",
        "document_type": "term_sheet_docx",
        "entities":      result,
        "errors":        errors,
    }

    if errors:
        print("Validation errors:")
        for e in errors:
            print(f"  WARNING: {e}")
    else:
        print("Validation passed.")

    return output


In [ ]:
output = build_output(entities, DOC_PATH)
print(json.dumps(output, indent=2))


Validation passed.
{
  "source": "ZF4894_ALV_07Aug2026_physical.docx",
  "engine": "rule_based_parser",
  "document_type": "term_sheet_docx",
  "entities": {
    "Counterparty": "BANK ABC",
    "Initial Valuation Date": "31 January 2025",
    "Notional": "EUR 1 million",
    "Valuation Date": "31 July 2026",
    "Maturity": "07 August 2026",
    "Underlying": "Allianz SE (ISIN DE0008404005, Reuters: ALVG.DE)",
    "Coupon": "0%",
    "Barrier": "75.00% of Shareini",
    "Calendar": "TARGET"
  },
  "errors": []
}
